# Automotive Dealership Profitability & Working Capital Analysis
## Exploratory Data Analysis & Financial Modeling Notebook

**Author:** Senior Financial Analyst & Corporate Finance Specialist  
**Dataset:** Vehicle Dataset from CarDekho (Kaggle) | **Population:** 6,926 Cleaned Vehicles  
**Analysis Baseline Year:** 2026  

> **Audit Governance:** Observed vehicle attributes are extracted from genuine CarDekho public listings. All profitability spreads, net unit margins, and working capital cycles represent separate corporate finance simulation models.

In [ ]:
# 1. Import Essential Analytics Libraries
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src directory to path
sys.path.append(os.path.abspath('../src'))
from data_loader import load_raw_data, inspect_raw_dataset
from data_cleaning import clean_dataset
from profitability_scenario import ProfitabilityScenarioModel
from working_capital import WorkingCapitalModel, compare_scenarios

sns.set_theme(style='whitegrid')
print('Libraries and custom modules successfully imported!')

## 2. Ingest and Inspect Cleaned Dataset

In [ ]:
df = pd.read_csv('../data/processed/cleaned_vehicle_data.csv')
print(f'Cleaned Dataset Dimensions: {df.shape[0]:,} rows, {df.shape[1]} columns')
df[['name', 'Vehicle_Brand', 'Vehicle_Model', 'selling_price', 'km_driven', 'Vehicle_Age', 'Fuel_Category']].head(5)

## 3. Marketplace Pricing & Distribution Statistics

In [ ]:
price_stats = df['selling_price'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99])
print('--- Selling Price Statistics (INR) ---')
for k, v in price_stats.items():
    print(f'{k:10s}: ₹{v:,.2f}')

## 4. Top Automotive Brands by Volume and Mean Asking Price

In [ ]:
brand_summary = df.groupby('Vehicle_Brand').agg(
    listings=('selling_price', 'count'),
    avg_price=('selling_price', 'mean'),
    median_price=('selling_price', 'median'),
    total_market_value=('selling_price', 'sum')
).sort_values(by='listings', ascending=False).head(10)

brand_summary['market_share_pct'] = (brand_summary['listings'] / len(df)) * 100
brand_summary

## 5. Dealership Profitability Scenario Simulation

In [ ]:
model = ProfitabilityScenarioModel({
    'acquisition_cost_pct': 0.82,
    'discount_pct': 0.04,
    'reconditioning_cost': 15000.0,
    'transportation_cost': 5000.0,
    'other_selling_costs': 8000.0
})
df_sim = model.calculate_vehicle_profitability(df)
print(f'Total Simulated Revenue: ₹{df_sim["Scenario_Revenue"].sum()/10000000:.2f} Crores')
print(f'Total Simulated Gross Profit: ₹{df_sim["Scenario_Gross_Profit"].sum()/10000000:.2f} Crores')
print(f'Portfolio Gross Margin: {df_sim["Scenario_Gross_Profit"].sum() / df_sim["Scenario_Revenue"].sum() * 100:.2f}%')
print(f'Total Net Contribution: ₹{df_sim["Scenario_Net_Contribution"].sum()/10000000:.2f} Crores')
print(f'Portfolio Net Margin: {df_sim["Scenario_Net_Contribution"].sum() / df_sim["Scenario_Revenue"].sum() * 100:.2f}%')

## 6. Corporate Working Capital Simulation & Cash Conversion Cycle

In [ ]:
wc_comp = compare_scenarios()
wc_comp